# Evaluate malignant-macrophage interactions

**Goal.** Show the recurrent ligand-receptor interactions between malignant cell states and
macrophage states, in both directions, from the cross-patient CellPhoneDB consensus.

**Input.** `cpdb_adata_per_patient/consensus_compartments_0.5_strict.h5ad`:
directed cell pairs × interactions, layers `means` (mean across patients), `fraction_significant`
(fraction of evaluable patients with P < 0.05; slots < 0.5 already set to 0) and `pvalues`.

**Panels.**
1. Heatmap: number of consensus interactions per sender → receiver pair.
2. Dot plots of the top interactions: dot size = fraction of patients significant; colour = consensus
   mean (raw) or mean scaled 0–1 per interaction across cell pairs.

**Output.** `cpdb_adata_per_patient/plots/consensus_50pct/malignant_macrophage/` (PDFs + source table).

In [ ]:
# !pip install --quiet scanpy seaborn
# %cd <project>/cpdb/cellphoneDB_v5_plot/OT

In [ ]:
import os

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

## Config

In [ ]:
group = "OT" ## on treatment (partial response + stable disease) # note: change this line to TN or TKIR patients 

consensus_label = "consensus_50pct"
input_path = "./cpdb_adata_per_patient/consensus_compartments_0.5_strict.h5ad" #set this to the directory either TN, OT or TKIR

out_dir = f"./cpdb_adata_per_patient/plots/{consensus_label}/malignant_macrophage"

malignant_states = ["1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like",
                    "4_Inflammatory EMT", "5_Neurod_EMT-like", "6_Proliferative"]     ## Scan all states first

macrophage_states = ["Macrophage INHBAhigh", "Macrophage CD36high", "Macrophage STAB1high"]

min_patients_per_slot = 1               # used only if the input has an 'n_patients' layer (1 = no filter)
top_n = 100                             # interactions ranked by mean consensus score across the cell pairs
cmap = "RdPu"
dot_scale = 200                         # marker area for fraction_significant = 1
size_legend = [0.5, 0.75, 1.0]          # input is already filtered at >= 0.5

os.makedirs(out_dir, exist_ok=True)

## 1. Load and subset malignant ↔ macrophage pairs

In [ ]:
adata = ad.read_h5ad(input_path)
s, r = adata.obs["sender"], adata.obs["receiver"]
mask = ((s.isin(malignant_states) & r.isin(macrophage_states))
        | (s.isin(macrophage_states) & r.isin(malignant_states)))
sub = adata[mask.to_numpy()].copy()


def layer_df(a, layer):
    return pd.DataFrame(a.layers[layer], index=a.obs_names, columns=a.var_names)


means = layer_df(sub, "means")
frac = layer_df(sub, "fraction_significant")
if "n_patients" in sub.layers:
    frac = frac.where(layer_df(sub, "n_patients") >= min_patients_per_slot, 0)
    means = means.where(frac > 0, 0)
print(f"{sub.n_obs} cell pairs x {sub.n_vars} interactions; "
      f"{int((frac > 0).values.sum())} consensus slots")

## 2. Heatmap: consensus interactions per sender → receiver

In [ ]:
order = malignant_states + macrophage_states
counts = (pd.DataFrame({"sender": sub.obs["sender"].astype(str), "receiver": sub.obs["receiver"].astype(str), "n": (frac > 0).sum(axis=1)})
          .pivot_table(index="sender", columns="receiver", values="n", aggfunc="sum", fill_value=0, observed=True))
counts = counts.reindex(index=[c for c in order if c in counts.index],
                        columns=[c for c in order if c in counts.columns], fill_value=0)

fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(counts, cmap="Reds", annot=True, fmt="d", linewidths=0.5, ax=ax,
            cbar_kws={"label": "No. consensus interactions (significant in >= 50% of patients)"})
ax.set(title=f"{group}: malignant ↔ macrophage", xlabel="Receiver", ylabel="Sender")
plt.savefig(f"{out_dir}/heatmap_malignant_macrophage_{consensus_label}.pdf", bbox_inches="tight")
plt.show()

## 3. Dot plots of top interactions

Top `top_n` interactions by mean consensus score across the malignant ↔ macrophage pairs; only
slots passing the consensus filter are drawn. Scaled colour = min–max of the mean per interaction
across the drawn cell pairs.

In [ ]:
direction_labels = {"malignant": "Malignant → Macrophage", "macrophage": "Macrophage → Malignant"}

pair_order = ([f"{m}|{k}" for m in malignant_states for k in macrophage_states]
              + [f"{k}|{m}" for k in macrophage_states for m in malignant_states])
top = means.mean(axis=0).nlargest(top_n).index
labels = sub.var["interacting_pair"]

plot_df = (means[top].stack().rename("mean_score").to_frame()
           .join(frac[top].stack().rename("frac_sig"))
           .rename_axis(["cell_pair", "interaction"]).reset_index())
plot_df = plot_df[plot_df["frac_sig"] > 0].copy()
plot_df["interaction_label"] = plot_df["interaction"].map(labels).fillna(plot_df["interaction"])
plot_df["sender"] = plot_df["cell_pair"].str.split("|").str[0]
plot_df["direction"] = np.where(plot_df["sender"].isin(malignant_states),
                                direction_labels["malignant"], direction_labels["macrophage"])
rng = plot_df.groupby("interaction_label")["mean_score"].agg(["min", "max"])
plot_df["scaled_score"] = ((plot_df["mean_score"] - plot_df["interaction_label"].map(rng["min"]))
                           / (plot_df["interaction_label"].map(rng["max"] - rng["min"]) + 1e-10))
plot_df["cell_pair"] = pd.Categorical(plot_df["cell_pair"],
                                      [p for p in pair_order if p in set(plot_df["cell_pair"])], ordered=True)
plot_df = plot_df.sort_values(["cell_pair", "mean_score"], ascending=[True, False])
plot_df.to_csv(f"{out_dir}/dotplot_source_{consensus_label}.csv", index=False)
print(f"{plot_df['cell_pair'].nunique()} cell pairs, {plot_df['interaction_label'].nunique()} interactions")

In [ ]:
def draw_dotplot(df, colour_col, cbar_label, title, path, vmin=None, vmax=None):
    pairs = list(df["cell_pair"].cat.remove_unused_categories().cat.categories)
    x = df["cell_pair"].cat.remove_unused_categories().cat.codes
    y_labels = list(dict.fromkeys(df["interaction_label"]))
    y = df["interaction_label"].map({l: i for i, l in enumerate(y_labels)})

    fig, ax = plt.subplots(figsize=(max(6, len(pairs) * 0.8), max(8, len(y_labels) * 0.32)))
    sc_ = ax.scatter(x, y, s=df["frac_sig"] * dot_scale, c=df[colour_col], cmap=cmap, vmin=vmin, vmax=vmax,
                     edgecolors="black", linewidths=0.5)
    plt.colorbar(sc_, ax=ax, label=cbar_label, shrink=0.4, pad=0.02)

    n_mal = sum(p.split("|")[0] in malignant_states for p in pairs)
    if 0 < n_mal < len(pairs):
        ax.axvline(n_mal - 0.5, color="black", lw=1.2, ls="--", alpha=0.6)
    for (start, stop), text in zip([(0, n_mal), (n_mal, len(pairs))], direction_labels.values()):
        if stop > start:
            ax.annotate(text, xy=((start + stop) / 2 - 0.5, 1), xycoords=("data", "axes fraction"),
                        xytext=(0, 10), textcoords="offset points", ha="center", fontsize=12, style="italic")

    handles = [ax.scatter([], [], s=v * dot_scale, c="grey", label=f"{v:.2f}") for v in size_legend]
    ax.add_artist(ax.legend(handles=handles, title="Fraction of patients\nsignificant", loc="upper left",
                            bbox_to_anchor=(1.15, 1), fontsize=8, title_fontsize=8))
    ax.set_xticks(range(len(pairs)), pairs, rotation=90, fontsize=12)
    ax.set_yticks(range(len(y_labels)), y_labels, fontsize=12)
    ax.invert_yaxis()
    ax.set(xlabel="Cell pair (sender | receiver)", ylabel="Ligand–receptor interaction")
    ax.set_title(title, fontsize=11, pad=24)
    plt.savefig(path, bbox_inches="tight", dpi=150)
    plt.show()


draw_dotplot(plot_df, "mean_score", "Consensus mean interaction score",
             f"{group}: malignant ↔ macrophage, mean score ({consensus_label})",
             f"{out_dir}/dotplot_raw_{consensus_label}.pdf", vmin=0, vmax=plot_df["mean_score"].max()) #Evaluate these plots and display relevant ints 

draw_dotplot(plot_df, "scaled_score", "Mean score scaled per interaction",
             f"{group}: malignant ↔ macrophage, scaled score ({consensus_label})",
             f"{out_dir}/dotplot_scaled_{consensus_label}.pdf", vmin=0, vmax=1) #Evaluate these plots and display relevant ints 